In [414]:
import numpy as np
import pandas as pd
import plotly.express as px

# 1

Implement rejection sampling to obtain 1000 samples distributed with a Beta(2,2) distributions using samples from a Uniform(0,1) distribution. Plot a histogram with your 1000 samples and super-impose the density of a Beta(2,2) distribution. Include all the code you used.

In [415]:
rng = np.random.default_rng(seed = 42)
n: int = 1_000

In [416]:
def beta_2_2(x):
    return 6 * x * (1 - x)

In [417]:
c: float = 1.5
accepted = []

while len(accepted) < n: 
    y = rng.uniform(0, 1)
    u = rng.uniform(0, 1)

    if u <= beta_2_2(y) / c:
        accepted.append(y)

In [418]:
x = np.linspace(0, 1, 1000)

fig = px.histogram(
    accepted,
    title="Rejection Sampling Histogram with Beta(2,2) Overlay",
    nbins=20,
    histnorm="probability density",
)
fig.update_traces(
    name="Rejection Sampled",
    showlegend=True,
)

line = px.line(x=x, y=beta_2_2(x))
line.update_traces(
    name="Beta(2,2)", 
    showlegend=True,
    line=dict(color="red", width=3)
)

fig.add_traces(line.data)

fig.show()

# 6
In this exercise, you will explore empirically the two estimators from Exercise 5: 
$$\hat{\lambda_1} = \frac{n}{\sum{T_i}}$$

and

$$\hat{\lambda_2} = \sqrt{\frac{2n}{\sum{T_i^2}}}$$

(i) Choose $\lambda = 3$ and generate 10,000 samples of 10 random variables $T_{1},...,T_{10}$ with distribution $Exp(3)$. For each of the 10,000 samples, construct the two estimators for $\lambda$. This should give you 10,000 pairs of estimators for λ, each of them constructed using 10 independent draws. Based on these simulations estimate the bias, the variance, and the mean squared errors for each of the two estimators. How do they compare? Which estimator would you choose based on this analysis?

(ii) Plot two histograms: First, the 10,000 $\hat{\lambda_1}$ estimators you obtained in (i); second the 10,000 $\hat{\lambda_2}$ estimators you obtainedin (i). Do the histograms agree with the asymptotic distribution of the estimators we derived in class and in Exercise 5?

### 6 (i)

In [419]:
lambda_: float = 3.0
n: int = 10_000     # n samples
n_rvs: int = 10     # n random variables: T_1, ..., T_10
rng21 = np.random.default_rng(seed = 21)

In [420]:
# Sample 10 random variables, 10000 times (vectorized)

samples = rng21.exponential(scale = 1 / lambda_, size = (n, n_rvs))

In [421]:
# Define estimators

def estimator_one(t: np.ndarray):
    return 1 / np.mean(t)

def estimator_two(t: np.ndarray):
    return np.sqrt((2 * n_rvs) / np.dot(t, t))

In [422]:
# Apply our estimators to each of our 10 random variables, for the 10_000 samples (vectorized)

samples_w_estimator_one = np.apply_along_axis(func1d=estimator_one, axis=1, arr=samples)
samples_w_estimator_two = np.apply_along_axis(func1d=estimator_two, axis=1, arr=samples)

samples_w_estimator_one[:5]

array([1.61424176, 3.2597449 , 2.83131974, 5.99191022, 5.429108  ])

In [423]:
# Define metrics to evaluate our estimators

def bias(estimates: np.ndarray, true_val: float) -> float:
    return np.mean(estimates) - true_val

def variance(estimates: np.ndarray) -> float:
    return np.var(estimates, ddof = 1)

def mse(estimates: np.ndarray, true_val: float) -> float:
    return np.mean((estimates - true_val)**2)

In [424]:
# Apply our metrics to our estimators

results = pd.DataFrame(
    data = {
        'bias' : [bias(samples_w_estimator_one, lambda_), bias(samples_w_estimator_two, lambda_)],
        'var'  : [variance(samples_w_estimator_one)     , variance(samples_w_estimator_two)],
        'mse'  : [mse(samples_w_estimator_one, lambda_) , mse(samples_w_estimator_two, lambda_)],
    },
    index = ['estimator_one', 'estimator_two']
)

results

,bias,var,mse
estimator_one,0.294685,1.351528,1.438231
estimator_two,0.517459,1.698092,1.965686


### Comment:

$\hat{\lambda_1}$ has lower bias, variance, and mean squared error than $\hat{\lambda_2}$, signaling that it is the superior estimator for $\lambda$. Hence, I would choose $\hat{\lambda_1}$ as an estimate for $\lambda$.

### 6 (ii)

In [425]:
fig1 = px.histogram(
    samples_w_estimator_one,
    title="Estimates of Lambda Using (1)",
    nbins=100,
    labels={"value": "lambda_hat_1 estimate"},
)
fig1.update_layout(showlegend=False)

fig1.show()

In [426]:
fig2 = px.histogram(
    samples_w_estimator_two,
    title="Estimates of Lambda Using (2)",
    nbins=100,
    labels={"value": "lambda_hat_2 estimate"},
)
fig2.update_layout(showlegend=False)

fig2.show()

### Comment:

- Both plots look slightly normal, but with long right tails. I assume that as $n \rightarrow \inf$ the distributions will more closely resemble the normal distributions we derived in class.

- Also, plot two seems to have slightly more variance, which agrees with the metrics we calculated.